# Clase 5 — Aprendizaje estructurado y adversarial con TensorFlow

**Objetivos de la clase**

- Explicar qué aporta el framework Neural Structured Learning (NSL, Aprendizaje Estructurado Neuronal) frente al entrenamiento estándar
- Entrenar un modelo usando grafos naturales y grafos sintetizados como señal adicional de aprendizaje
- Aplicar regularización adversarial para mejorar la robustez de un clasificador de imágenes

Este notebook es la guía de la clase en vivo. El manual (`07_aprendizaje_estructurado_adversarial.md`) tiene la explicación completa; aquí vamos directos al código.


## 1. ¿Qué aporta NSL frente al entrenamiento estándar?

En el entrenamiento estándar, la función de pérdida (loss) solo compara la predicción del modelo con la etiqueta real de cada ejemplo, uno por uno. Neural Structured Learning (NSL) añade un **término de regularización extra** a esa pérdida, calculado a partir de la relación entre ejemplos:

- **Regularización de grafos**: el modelo aprende a dar predicciones parecidas para ejemplos conectados en un grafo (`nsl.keras.GraphRegularization`)
- **Regularización adversarial**: el modelo aprende a dar predicciones parecidas para un ejemplo y una versión de ese ejemplo perturbada a propósito (`nsl.keras.AdversarialRegularization`)

En los dos casos, la mecánica es la misma: construyes un modelo base normal de Keras y lo envuelves con una clase de NSL. No hace falta rediseñar la arquitectura.


## 2. Instalación

`neural_structured_learning` no viene instalado por defecto en Colab, así que lo instalamos primero.


In [ ]:
# Instala Neural Structured Learning (NSL) si no está disponible
!pip install --quiet neural-structured-learning


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import neural_structured_learning as nsl  # convención habitual: abreviar a "nsl"
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)


## 3. Grafos naturales: el dataset Cora

Un **grafo natural** usa relaciones que ya existen en los datos. Cora es un grafo de citas: los nodos son artículos científicos y las aristas son citas entre ellos. La tarea es clasificar cada artículo en una de siete categorías, y NSL aprovecha la idea de que "si A cita a B, probablemente tratan un tema parecido".

El flujo con NSL:

1. Aumentar los datos con las características de los vecinos en el grafo (`pack_nbrs` / el script `preprocess_cora_dataset.py` en este ejemplo)
2. Definir un modelo base (aquí, un perceptrón multicapa, MLP)
3. Envolver el modelo con `nsl.keras.GraphRegularization`
4. Compilar, entrenar y evaluar igual que cualquier modelo de Keras

Descargamos el dataset y lo preprocesamos con el script oficial de NSL (necesita conexión a internet, disponible en Colab).


In [ ]:
# Descarga el dataset Cora (grafo de citas) y el script de preprocesado de NSL
!wget --quiet -P /tmp https://linqs-data.soe.ucsc.edu/public/lbc/cora.tgz
!tar -C /tmp -xzf /tmp/cora.tgz
!wget --quiet https://raw.githubusercontent.com/tensorflow/neural-structured-learning/master/neural_structured_learning/examples/preprocess/cora/preprocess_cora_dataset.py

# El script genera los ejemplos de entrenamiento ya aumentados con sus vecinos del grafo
!python preprocess_cora_dataset.py \
  --input_cora_content=/tmp/cora/cora.content \
  --input_cora_graph=/tmp/cora/cora.cites \
  --max_nbrs=5 \
  --output_train_data=/tmp/cora/train_merged_examples.tfr \
  --output_test_data=/tmp/cora/test_examples.tfr


In [ ]:
NBR_FEATURE_PREFIX = 'NL_nbr_'
NBR_WEIGHT_SUFFIX = '_weight'

class HParamsGraph(object):
    def __init__(self):
        self.num_classes = 7
        self.max_seq_length = 1433          # tamaño del vocabulario (bag-of-words)
        self.distance_type = nsl.configs.DistanceType.L2
        self.graph_regularization_multiplier = 0.1  # peso del término de grafo en la pérdida
        self.num_neighbors = 1
        self.num_fc_units = [50, 50]
        self.train_epochs = 20               # en el tutorial original: 100 (aquí lo reducimos para ir más rápido en clase)
        self.batch_size = 128
        self.dropout_rate = 0.5

HPARAMS = HParamsGraph()

def make_dataset(file_path, training=False):
    def parse_example(example_proto):
        feature_spec = {
            'words': tf.io.FixedLenFeature([HPARAMS.max_seq_length], tf.int64,
                                            default_value=tf.constant(0, dtype=tf.int64, shape=[HPARAMS.max_seq_length])),
            'label': tf.io.FixedLenFeature((), tf.int64, default_value=-1),
        }
        if training:
            for i in range(HPARAMS.num_neighbors):
                nbr_feature_key = '{}{}_{}'.format(NBR_FEATURE_PREFIX, i, 'words')
                nbr_weight_key = '{}{}{}'.format(NBR_FEATURE_PREFIX, i, NBR_WEIGHT_SUFFIX)
                feature_spec[nbr_feature_key] = tf.io.FixedLenFeature(
                    [HPARAMS.max_seq_length], tf.int64,
                    default_value=tf.constant(0, dtype=tf.int64, shape=[HPARAMS.max_seq_length]))
                feature_spec[nbr_weight_key] = tf.io.FixedLenFeature([1], tf.float32, default_value=tf.constant([0.0]))
        features = tf.io.parse_single_example(example_proto, feature_spec)
        label = features.pop('label')
        return features, label

    dataset = tf.data.TFRecordDataset([file_path])
    if training:
        dataset = dataset.shuffle(10000)
    dataset = dataset.map(parse_example)
    dataset = dataset.batch(HPARAMS.batch_size)
    return dataset

train_dataset = make_dataset('/tmp/cora/train_merged_examples.tfr', training=True)
test_dataset = make_dataset('/tmp/cora/test_examples.tfr')


In [ ]:
def make_mlp_functional_model(hparams):
    """Perceptron multicapa (MLP) sencillo: el modelo base de siempre, sin nada especial de NSL."""
    inputs = tf.keras.Input(shape=(hparams.max_seq_length,), dtype='int64', name='words')
    cur_layer = tf.keras.layers.Lambda(lambda x: tf.keras.backend.cast(x, tf.float32))(inputs)
    for num_units in hparams.num_fc_units:
        cur_layer = tf.keras.layers.Dense(num_units, activation='relu')(cur_layer)
        cur_layer = tf.keras.layers.Dropout(hparams.dropout_rate)(cur_layer)
    outputs = tf.keras.layers.Dense(hparams.num_classes)(cur_layer)
    return tf.keras.Model(inputs, outputs=outputs)

# --- Modelo base, SIN regularización de grafos ---
base_model = make_mlp_functional_model(HPARAMS)
base_model.compile(optimizer='adam',
                    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=['accuracy'])
base_model.fit(train_dataset, epochs=HPARAMS.train_epochs, verbose=0)
base_eval = dict(zip(base_model.metrics_names, base_model.evaluate(test_dataset, verbose=0)))
print('Accuracy modelo base (sin grafo):', base_eval['accuracy'])


In [ ]:
# --- Mismo modelo, envuelto con GraphRegularization ---
graph_reg_config = nsl.configs.make_graph_reg_config(
    max_neighbors=HPARAMS.num_neighbors,
    multiplier=HPARAMS.graph_regularization_multiplier,
    distance_type=HPARAMS.distance_type,
    sum_over_axis=-1)

graph_model = nsl.keras.GraphRegularization(make_mlp_functional_model(HPARAMS), graph_reg_config)
graph_model.compile(optimizer='adam',
                     loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                     metrics=['accuracy'])
graph_model.fit(train_dataset, epochs=HPARAMS.train_epochs, verbose=0)
graph_eval = dict(zip(graph_model.metrics_names, graph_model.evaluate(test_dataset, verbose=0)))
print('Accuracy modelo con regularización de grafos:', graph_eval['accuracy'])


El modelo con regularización de grafos suele superar al modelo base porque aprovecha las citas entre artículos, incluyendo la información de artículos que no llevan etiqueta explícita en el lote de entrenamiento.


## 4. Grafos sintetizados: cuando no hay un grafo de serie

La mayoría de los datasets no traen un grafo explícito (opiniones de películas, imágenes, ventas...). Para esos casos, NSL permite **sintetizar** un grafo a partir de la similitud entre ejemplos:

1. Convertir cada ejemplo en un embedding (representación numérica densa) — en el tutorial oficial de IMDb se usan embeddings preentrenados de TensorFlow Hub (Swivel)
2. Calcular la similitud entre pares de embeddings (similitud coseno) con `nsl.tools.build_graph_from_config`, y conectar dos ejemplos si superan un umbral
3. A partir de ahí, el flujo es idéntico al de un grafo natural: aumentar con vecinos (`pack_nbrs`) y envolver el modelo con `GraphRegularization`

Para no depender de una descarga pesada de TensorFlow Hub en clase, reproducimos el paso 2 (la parte conceptualmente nueva) a pequeña escala con NumPy, sobre unos embeddings de juguete.


In [ ]:
# Simulamos 6 "documentos" ya convertidos a embeddings (en un caso real vendrían
# de un modelo preentrenado, p. ej. Swivel o BERT, aplicado sobre el texto real)
np.random.seed(0)
embeddings = {
    'doc_0': np.array([0.9, 0.1, 0.0]),
    'doc_1': np.array([0.85, 0.15, 0.0]),   # parecido a doc_0
    'doc_2': np.array([0.0, 0.9, 0.1]),
    'doc_3': np.array([0.05, 0.92, 0.1]),   # parecido a doc_2
    'doc_4': np.array([0.1, 0.0, 0.9]),
    'doc_5': np.array([0.0, 0.1, 0.88]),    # parecido a doc_4
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarity_threshold = 0.99  # el mismo valor que usa el tutorial oficial de IMDb
edges = []
names = list(embeddings.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        sim = cosine_similarity(embeddings[names[i]], embeddings[names[j]])
        if sim >= similarity_threshold:
            edges.append((names[i], names[j], round(float(sim), 4)))

print('Grafo sintetizado (aristas por encima del umbral de similitud):')
for edge in edges:
    print(' ', edge)


Fíjate en que el grafo resultante conecta `doc_0` con `doc_1`, `doc_2` con `doc_3` y `doc_4` con `doc_5`: exactamente los pares que construimos para que fueran similares. En un proyecto real, `nsl.tools.build_graph_from_config` hace este mismo cálculo de forma eficiente sobre miles de embeddings guardados en formato `TFRecord`, y el resultado se combina con las características originales mediante `nsl.tools.pack_nbrs`, igual que en el caso de Cora.


## 5. Regularización adversarial: robustez frente a ataques

Un **ejemplo adversarial** es una versión de un ejemplo original con una perturbación mínima (casi imperceptible) calculada específicamente para confundir al modelo, siguiendo la dirección que más aumenta su error. NSL trata cada ejemplo adversarial como un vecino del original: es una forma de construir "estructura" incluso sin ningún grafo.

Vamos a comprobarlo con MNIST: entrenamos un modelo base y uno con regularización adversarial, y medimos cómo se comporta cada uno frente a imágenes perturbadas.


In [ ]:
class HParamsAdv(object):
    def __init__(self):
        self.input_shape = [28, 28, 1]
        self.num_classes = 10
        self.conv_filters = [32, 64, 64]
        self.kernel_size = (3, 3)
        self.pool_size = (2, 2)
        self.num_fc_units = [64]
        self.batch_size = 32
        self.epochs = 3                # en el tutorial original: 5 (reducido para ir más rápido en clase)
        self.adv_multiplier = 0.2      # peso del término adversarial en la pérdida
        self.adv_step_size = 0.2       # magnitud de la perturbación
        self.adv_grad_norm = 'infinity'

HPARAMS_ADV = HParamsAdv()

IMAGE_INPUT_NAME = 'image'
LABEL_INPUT_NAME = 'label'

datasets = tfds.load('mnist')
train_dataset_adv = datasets['train']
test_dataset_adv = datasets['test']

def normalize(features):
    features[IMAGE_INPUT_NAME] = tf.cast(features[IMAGE_INPUT_NAME], tf.float32) / 255.0
    return features

def convert_to_tuples(features):
    return features[IMAGE_INPUT_NAME], features[LABEL_INPUT_NAME]

def convert_to_dictionaries(image, label):
    return {IMAGE_INPUT_NAME: image, LABEL_INPUT_NAME: label}

train_dataset_adv = train_dataset_adv.map(normalize).shuffle(10000).batch(HPARAMS_ADV.batch_size).map(convert_to_tuples)
test_dataset_adv = test_dataset_adv.map(normalize).batch(HPARAMS_ADV.batch_size).map(convert_to_tuples)


In [ ]:
def build_base_model(hparams):
    """CNN sencilla: 3 capas convolucionales + 2 densas, con la Functional API de Keras."""
    inputs = tf.keras.Input(shape=hparams.input_shape, dtype=tf.float32, name=IMAGE_INPUT_NAME)
    x = inputs
    for i, num_filters in enumerate(hparams.conv_filters):
        x = tf.keras.layers.Conv2D(num_filters, hparams.kernel_size, activation='relu')(x)
        if i < len(hparams.conv_filters) - 1:
            x = tf.keras.layers.MaxPooling2D(hparams.pool_size)(x)
    x = tf.keras.layers.Flatten()(x)
    for num_units in hparams.num_fc_units:
        x = tf.keras.layers.Dense(num_units, activation='relu')(x)
    pred = tf.keras.layers.Dense(hparams.num_classes)(x)
    return tf.keras.Model(inputs=inputs, outputs=pred)

# --- Modelo base, SIN regularización adversarial ---
base_model_mnist = build_base_model(HPARAMS_ADV)
base_model_mnist.compile(optimizer='adam',
                          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                          metrics=['acc'])
base_model_mnist.fit(train_dataset_adv, epochs=HPARAMS_ADV.epochs, verbose=0)
base_results = dict(zip(base_model_mnist.metrics_names, base_model_mnist.evaluate(test_dataset_adv, verbose=0)))
print('Accuracy modelo base sobre datos limpios:', base_results['acc'])


In [ ]:
# --- Mismo modelo, envuelto con AdversarialRegularization ---
adv_config = nsl.configs.make_adv_reg_config(
    multiplier=HPARAMS_ADV.adv_multiplier,
    adv_step_size=HPARAMS_ADV.adv_step_size,
    adv_grad_norm=HPARAMS_ADV.adv_grad_norm)

base_adv_model = build_base_model(HPARAMS_ADV)
adv_model = nsl.keras.AdversarialRegularization(
    base_adv_model, label_keys=[LABEL_INPUT_NAME], adv_config=adv_config)

# AdversarialRegularization necesita la etiqueta junto a la imagen, así que
# convertimos el dataset de tuplas (image, label) a diccionarios {'image':..., 'label':...}
train_set_for_adv_model = train_dataset_adv.map(convert_to_dictionaries)
test_set_for_adv_model = test_dataset_adv.map(convert_to_dictionaries)

adv_model.compile(optimizer='adam',
                   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                   metrics=['acc'])
adv_model.fit(train_set_for_adv_model, epochs=HPARAMS_ADV.epochs, verbose=0)
adv_results = dict(zip(adv_model.metrics_names, adv_model.evaluate(test_set_for_adv_model, verbose=0)))
print('Accuracy modelo adversarial-regularizado sobre datos limpios:', adv_results['sparse_categorical_accuracy'])


### Robustez frente a ejemplos perturbados

Ahora generamos ejemplos adversariales a partir del modelo base (con `perturb_on_batch`) y comparamos cómo se comportan el modelo base y el modelo adversarial-regularizado frente a esas mismas imágenes perturbadas.


In [ ]:
# Usamos el modelo base para generar las perturbaciones (referencia)
reference_model = nsl.keras.AdversarialRegularization(
    base_model_mnist, label_keys=[LABEL_INPUT_NAME], adv_config=adv_config)
reference_model.compile(optimizer='adam',
                         loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                         metrics=['acc'])

models_to_eval = {
    'base': base_model_mnist,
    'adv-regularized': adv_model.base_model,
}
metrics = {name: tf.keras.metrics.SparseCategoricalAccuracy() for name in models_to_eval}

for batch in test_set_for_adv_model:
    perturbed_batch = reference_model.perturb_on_batch(batch)
    perturbed_batch[IMAGE_INPUT_NAME] = tf.clip_by_value(perturbed_batch[IMAGE_INPUT_NAME], 0.0, 1.0)
    y_true = perturbed_batch.pop(LABEL_INPUT_NAME)
    for name, model in models_to_eval.items():
        y_pred = model(perturbed_batch)
        metrics[name](y_true, y_pred)

for name, metric in metrics.items():
    print('Accuracy del modelo %s frente a imágenes perturbadas: %f' % (name, metric.result().numpy()))


Deberías ver que el modelo base pierde mucha precisión frente a las imágenes perturbadas, mientras que el modelo con regularización adversarial la mantiene mucho mejor. Esa diferencia es el efecto que estás midiendo cada vez que ajustas `adv_multiplier`: cuanto mayor es, más peso le da el entrenamiento a resistir esas perturbaciones (a costa, potencialmente, de un poco de precisión sobre datos limpios).


## Resumen de la clase

- NSL añade un término de regularización a la pérdida estándar de Keras: por grafo (`GraphRegularization`) o por perturbación adversarial (`AdversarialRegularization`)
- Un grafo natural usa relaciones que ya existen en los datos (citas, como en Cora); un grafo sintetizado se construye a partir de la similitud entre embeddings cuando no hay relación explícita
- La regularización adversarial entrena con vecinos generados dinámicamente (ejemplos perturbados), sin necesitar ningún grafo previo
- En los tres casos, el modelo base de Keras no cambia: solo lo envuelves con una clase de NSL y sigues con el flujo compilar → entrenar → evaluar

Ahora abre el notebook de práctica de esta clase para experimentar tú mismo con los hiperparámetros.
